In [2]:
import numpy
print(numpy.__version__)

2.5.1


## Projeto: "Convergência Estatística Aplicada à Precificação de Opções via Modelo Binomial"
Demonstrar empiricamente, através de simulação, como a LFGN, a LGGN e o TLC (via Teorema de De Moivre-Laplace) sustentam o método de precificação de opções por árvore binomial — e quantificar a incerteza da estimativa via Monte Carlo.

In [3]:
import numpy as np
import matplotlib.pyplot as mt
from scipy.stats import binom , norm
import yfinance as yf
import pandas as pd
import requests as rt
import math as m 

Obtenção dos dados que seram usados

In [4]:
dados = yf.download(
    "PETR4.SA",
    period="2y"
)



[*********************100%***********************]  1 of 1 completed


## Modulo  0: preparação e coleta dos dados

Calcular o Log-retorno diario

In [5]:
close  = dados['Close'].squeeze()
razao = close / close.shift(1) # razao entre o fechamento de hoje com o  dia anterior


log = np.log(razao) # calcula o log-retorno diario
log_retorno = log.dropna() 

dp = log_retorno.std() # desvio padrao
dp_anual = dp * np.sqrt(252) # volatilidade anual





print(dp_anual)



0.24555698192822076


Puxando API do banco central

In [6]:
codigo = 11
n = 1
url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados/ultimos/{n}?formato=json'
resposta = rt.get(url)
selic = resposta.json()
r = float(selic[0]['valor']) # Taxa livre de risco



Extraindo os restante dos dados

In [7]:
s0 = close.iat[-1] # Preço Atual da ação
strike = round(s0) # Preço em exercicio, arredondado
T = 0.25 # tempo de vencimento do mercado
n_steps = 200 # Divide em quantos remos vai ter ate a data de vencimento


OUTPUTS

In [8]:

def outputs(vol):
    deltaT = T/n_steps
    u = m.exp(vol * m.sqrt(deltaT))
    d = 1/u
    p = (m.exp(r * deltaT) - d) / (u - d)

    return u,d, p

u, d, p = outputs(dp_anual)

print(u, d, p)


1.0087195460474783 0.9913558272151615 0.5016113641819253


Função  simuladora de trajetoria

In [9]:
# Achando o valor de k 

def simulacao(s, P, sub, des, si, K_strike):
    k = np.random.binomial(s, P)
    Sn = si * (sub ** k) * (des ** (s - k))
    payoff = max(Sn - K_strike, 0)

    return Sn, k, payoff

Sn, k, payoff = simulacao(n_steps, p, u, d, s0, strike)

print(Sn, k, payoff)


43.610698774744804 101 0.6106987747448045


## Calculando o preço teorico


In [10]:
esperanca = 0

for k in range(201):
    probk = binom.pmf(k, n_steps, p)  # probabilidade de k altas

    Snk = s0 * (u ** k) * (d ** (n_steps - k))

    payoffk = max(Snk - strike, 0)

    contribuicao = probk * payoffk

    esperanca += contribuicao


def teorico(j, t, e):
    C = np.exp(-j * t) * e
    return C


preco_teorico = teorico(r, T, esperanca)

print(preco_teorico)




2.3055118244713237


## Modulo 2: Lei Fraca dos Grandes Numeros

# Simulação 

In [ ]:
payoff_N = []

for m in range(1001):
    _, _, pyf= simulacao(n_steps, p, u, d, s0, strike)
    payoff_N.append(float(pyf))

media_payoff = np.mean(payoff_N)

Cn = m.exp()


print(payoff_N)
print(media_payoff)

[1.3745455095067527, 0.0, 3.747300363432977, 0.0, 1.3745455095067527, 3.747300363432977, 0.0, 0.0, 2.1517711134590343, 0.0, 0.0, 0.0, 0.0, 1.3745455095067527, 0.0, 0.0, 0.0, 4.566085059499876, 2.1517711134590343, 5.399210869883554, 10.713507892333979, 0.0, 0.6106987747448045, 3.747300363432977, 1.3745455095067527, 0.0, 7.109494978662127, 5.399210869883554, 0.0, 0.6106987747448045, 0.0, 0.0, 2.1517711134590343, 0.0, 0.0, 2.942609919135485, 0.6106987747448045, 4.566085059499876, 0.0, 0.0, 5.399210869883554, 0.0, 0.0, 14.576731342600645, 9.788903771997035, 0.0, 7.987168925546619, 0.6106987747448045, 1.3745455095067527, 0.0, 2.1517711134590343, 0.0, 0.0, 14.576731342600645, 2.1517711134590343, 17.655421325737343, 0.0, 8.880215439194522, 1.3745455095067527, 2.942609919135485, 2.942609919135485, 0.0, 2.1517711134590343, 0.0, 0.0, 0.0, 1.3745455095067527, 7.109494978662127, 5.399210869883554, 0.0, 0.0, 0.0, 1.3745455095067527, 0.0, 0.0, 0.0, 0.0, 5.399210869883554, 0.0, 6.246928980959204, 0.0